In [156]:
def pp(msg=None, value=None):
    if msg is not None:
        print(msg)
    if value is not None:
        print(value)
    print('=' * 100)

In [157]:
import pandas as pd

flights_raw = pd.read_csv('data/flights.csv')
flights = flights_raw.copy()
pp("flights.csv", flights.shape)

pp("Columns", flights.columns)
pp("Dtypes", flights.dtypes)

flights.csv
(20000, 5)
Columns
Index(['date', 'delay', 'distance', 'origin', 'destination'], dtype='str')
Dtypes
date             str
delay          int64
distance       int64
origin           str
destination      str
dtype: object


Every row is a single flight from origin to destination airport:

In [158]:
pp("Sample flight", flights.head(1))

Sample flight
                  date  delay  distance origin destination
0  2001-01-01 00:47:00     66      1750    DTW         LAS


# Data cleaning
Data problems:
1. `date` as string
2. `distance` unit not specified 
3. `delay` has negative values

In [159]:
# 1. `date` as string
# Convert `date` str to datetime using Pandas (20000 rows):
flights['date'] = pd.to_datetime(flights['date'])
flights.dtypes


date           datetime64[us]
delay                   int64
distance                int64
origin                    str
destination               str
dtype: object

In [160]:
# 2. `distance` unit not specified 
# Considering USA flight data, we can assume miles as distance unit.
# Let's make a quick check of distances for first 3 flights (https://www.greatcirclemap.com/):
# DTW -> LAS = 1749 miles
# HNL -> SFO = 2398 miles
# LAS -> OAK = 407 miles
# for distance in flights.head(3)['distance']:
#     print(distance)
# # 1750
# # 2399
# # 407
# As assumption is correct, let's rename the column:
flights.rename(columns={'distance': 'distance_miles'}, inplace=True)
flights.columns

Index(['date', 'delay', 'distance_miles', 'origin', 'destination'], dtype='str')

In [161]:
# 3. `delay` has negative values
# Negative/zero delay means early/no delay arrival.
# A separate boolean value can be useful for further evaluations (20000 rows):
flights['is_delayed'] = flights['delay'] > 0
pp("Total of flights", len(flights))
flights['is_delayed'].value_counts(normalize=True)

Total of flights
20000


is_delayed
False    0.52535
True     0.47465
Name: proportion, dtype: float64

In [162]:
# As the description says that `delay` is in minutes, let's adjust the corresponding column name:
flights.rename(columns={'delay': 'delay_minutes'}, inplace=True)
# And save cleaned data:
flights.to_csv('data/flights_clean.csv', index=False)
flights.dtypes

date              datetime64[us]
delay_minutes              int64
distance_miles             int64
origin                       str
destination                  str
is_delayed                  bool
dtype: object